# Figure 4: Total memory depth and its allocation

For every total depth $K=k_1+k_2\le 8$ and every allocation $(k_1,k_2)$ with full readout ($\alpha=\beta=1$), the
figure shows the mean long-run cooperation $\bar\rho$ over 512 area-uniform initial strategy compositions
(shifted Halton sample, seed 2026), at $b/c=5$ and $e_1=e_2=0.02$. The black envelope marks the allocation with the
largest $\bar\rho$ at each $K$. Run from this directory. Saved data are used by default; `RECOMPUTE=True` regenerates
all 180 conditions with the verified engine in `../engine` (about 10–20 minutes on one core).

## Configuration

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

ROOT = Path.cwd()
ENGINE = (ROOT / '..' / 'engine').resolve()
sys.path.insert(0, str(ENGINE))
DATA = ROOT / 'data' / 'figure4'
FIGURES = ROOT / 'figures' / 'figure4'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
RECOMPUTE = False

NORM_ORDER = ('SC', 'SH', 'SJ', 'SS')
NORM_NAMES = {'SC': 'Scoring', 'SH': 'Shunning', 'SJ': 'Stern Judging', 'SS': 'Simple Standing'}
B_OVER_C = 5.0
E1 = E2 = 0.02
SEED = 2026
K_MAX = 8
N_INITIAL = 512

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.linewidth': .7, 'xtick.direction': 'out',
                     'ytick.direction': 'out', 'pdf.fonttype': 42, 'svg.fonttype': 'none'})

## Numerical method and outcome classification

Reputations at each strategy composition are the equilibrium reached by relaxing the auxiliary reputation flow
from the all-good state (main model convention). The engine (`../engine/irsim.c`) tabulates this map on a
triangular lattice of 7,381 nodes ($M=120$) and interpolates it linearly only inside lattice cells where it is smooth;
in every cell in which a reputation component varies by more than 0.02 (cells crossed by a fold or a steep
transition) it is evaluated exactly at the requested composition. The replicator field is therefore never
interpolated across a discontinuity of the selected reputation branch. Trajectories are integrated with the explicit
midpoint rule ($\Delta t=0.2$) until realized cooperation changes by at most $10^{-7}$ over three consecutive
intervals of 100 time units (cap: $4\times10^4$ time units).

Each trajectory converges either to an attractor with positive cooperation or to the zero-cooperation set (the ALLD
vertex and the low-reputation segment of the ALLD-DISC edge, along which the dynamics drift towards ALLD). The
cooperative basin $\omega_C$ is the fraction of initial compositions in the former class;
$\bar\rho_C$ and $\bar\rho_D$ are the mean long-run cooperation within each class, so that
$\bar\rho=\omega_C\bar\rho_C+(1-\omega_C)\bar\rho_D$ (definitions and the gap check in `../engine/analysis.py`).

In [2]:
import irsim, analysis

def compute_condition(norm, k1, k2):
    p = irsim.params(norm, k1, k2, 1.0, 1.0, E1, E2, B_OVER_C, 1.0)
    out, info = irsim.run(p, irsim.halton_shifted(N_INITIAL, SEED), bs=(B_OVER_C,))
    o = out[0]; rho = (1 - E1) * o[:, 6]
    row = dict(norm=norm, k1=k1, k2=k2, K=k1 + k2, alpha=1.0, beta=1.0, b_over_c=B_OVER_C, e1=E1, e2=E2,
               sample='halton2026', **analysis.summarize_outcomes(rho),
               max_steps=int(o[:, 7].max()), n_step_cap=int((o[:, 7] >= 400 * 500).sum()),
               max_final_strategy_residual=float(o[:, 8].max()), lattice_max_residual=info[0],
               flagged_cells=int(info[1]), exact_evaluations=int(info[2]))
    inv = analysis.pure_disc_invasion(norm, k1, k2, 1.0, 1.0, B_OVER_C, E1, E2)
    row.update(lambda_ALLD=inv['lambda_ALLD'], lambda_ALLC=inv['lambda_ALLC'],
               disc_locally_protected=bool(analysis.locally_protected(inv['lambda_ALLD'], inv['lambda_ALLC'])))
    return row, o

# identity checks of the stationary response used throughout
u = np.linspace(0., 1., 101)
np.testing.assert_allclose(analysis.response(u, 0, 0), u, atol=1e-14)
np.testing.assert_allclose(analysis.response(u, 1, 1), u**2 * (2 - u) / (1 - u + u**2), atol=1e-14)
print('Response-function identities passed.')

Response-function identities passed.


## Data

In [3]:
path = DATA / 'total_depth_results.csv'
if RECOMPUTE:
    rows, finals = [], []
    for norm in NORM_ORDER:
        for K in range(K_MAX + 1):
            for k1 in range(K + 1):
                row, o = compute_condition(norm, k1, K - k1); rows.append(row); finals.append(o)
                print(f"{norm} ({k1},{K - k1}): mean cooperation {row['mean_cooperation']:.4f}", flush=True)
    data = pd.DataFrame(rows); data.to_csv(path, index=False)
    np.savez_compressed(DATA / 'total_depth_results_final_states.npz',
                        final_frequencies=np.array([o[:, :3] for o in finals], np.float32),
                        final_cooperation=np.array([(1 - E1) * o[:, 6] for o in finals], np.float32),
                        norm=data.norm.to_numpy(), k1=data.k1.to_numpy(), k2=data.k2.to_numpy())
data = pd.read_csv(path)
assert len(data) == 180 and not data.duplicated(['norm', 'k1', 'k2']).any()
for norm in NORM_ORDER:
    keys = set(map(tuple, data.loc[data.norm.eq(norm), ['K', 'k1', 'k2']].to_numpy(int)))
    assert keys == {(K, k1, K - k1) for K in range(K_MAX + 1) for k1 in range(K + 1)}
assert np.allclose(data[['b_over_c', 'e1', 'e2', 'alpha', 'beta']], [B_OVER_C, E1, E2, 1., 1.])
assert data.n_initial.eq(N_INITIAL).all() and data.outcome_gap_ok.all()
assert data.lattice_max_residual.max() <= 1e-10
with np.load(DATA / 'total_depth_results_final_states.npz', allow_pickle=True) as saved:
    np.testing.assert_allclose(saved['final_cooperation'].astype(float).mean(1), data.mean_cooperation, atol=1e-6)

def allocation_fraction(row):
    return .5 if int(row.K) == 0 else float(row.k1) / float(row.K)

best_rows = []
for (norm, K), part in data.groupby(['norm', 'K'], sort=False):
    ordered = part.sort_values(['mean_cooperation', 'k1'], ascending=[False, False])
    best = ordered.iloc[0]
    best_rows.append(dict(norm=norm, K=int(K), k1=int(best.k1), k2=int(best.k2),
                          mean_cooperation=float(best.mean_cooperation),
                          cooperative_basin=float(best.cooperative_basin),
                          cooperation_cooperative=float(best.cooperation_cooperative),
                          allocation_fraction=allocation_fraction(best),
                          gap_to_second=np.nan if len(ordered) == 1 else float(best.mean_cooperation - ordered.iloc[1].mean_cooperation),
                          allocations=len(ordered)))
best_allocations = pd.DataFrame(best_rows)
best_allocations.to_csv(DATA / 'best_allocations.csv', index=False)
display(best_allocations.round(4))
print('Step-cap hits (trajectories not meeting the convergence criterion within the cap):',
      data.loc[data.n_step_cap > 0, ['norm', 'k1', 'k2', 'n_step_cap']].to_dict('records'))

   norm  K  k1  k2  mean_cooperation  cooperative_basin  cooperation_cooperative  allocation_fraction  gap_to_second  allocations
0    SC  0   0   0            0.6101             0.6348                   0.9611               0.5000            NaN            1
1    SC  1   1   0            0.8064             0.8242                   0.9784               1.0000         0.8064            2
2    SC  2   2   0            0.8651             0.8828                   0.9800               1.0000         0.2740            3
3    SC  3   3   0            0.8786             0.8965                   0.9800               1.0000         0.1762            4
4    SC  4   4   0            0.8843             0.9023                   0.9800               1.0000         0.1550            5
5    SC  5   5   0            0.8881             0.9062                   0.9800               1.0000         0.1436            6
6    SC  6   6   0            0.8862             0.9043                   0.9800          

## Panels

The black envelope selects the largest $\bar\rho$ at each depth (ties favour larger $k_1$). Non-winning points have
small horizontal offsets for visibility. Labels give the winning $(k_1,k_2)$; the colour encodes $k_1/K$ (0.5 at
$K=0$). Differences between allocations smaller than about 0.01 are within the variation expected from the finite
sample of initial compositions and should not be read as rankings.

In [4]:
CMAP = LinearSegmentedColormap.from_list('allocation', ['#C9853D', '#E6E6E6', '#0A5A86'])
CNORM = Normalize(0, 1)
HANDLES = [Line2D([], [], linestyle='none', marker='o', color='#7095A8', markersize=4, label='All allocations'),
           Line2D([], [], color='#34383A', marker='o', markerfacecolor='#7095A8', linewidth=1.15, markersize=5,
                  label='Largest mean cooperation at each $K$')]

def spread_x(K, group, best_index):
    x = np.full(len(group), float(K)); k1 = group.k1.to_numpy(int); best_k1 = int(group.loc[best_index, 'k1'])
    left, right = np.flatnonzero(k1 < best_k1), np.flatnonzero(k1 > best_k1)
    if len(left): x[left] += np.linspace(-.1, -.1 / len(left), len(left))
    if len(right): x[right] += np.linspace(.1 / len(right), .1, len(right))
    return x

def draw_panel(ax, norm, panel=None):
    part = data.loc[data.norm.eq(norm)].sort_values(['K', 'k1', 'k2'])
    for K, group in part.groupby('K', sort=True):
        best = group.sort_values(['mean_cooperation', 'k1'], ascending=[False, False]).iloc[0]
        values = group.mean_cooperation.to_numpy(float)
        ax.vlines(K, values.min(), values.max(), color='#D9E0E3', linewidth=.75, linestyles=(0, (2.2, 2.2)), zorder=1)
        ax.scatter(spread_x(K, group, best.name), values, c=[allocation_fraction(r) for r in group.itertuples()],
                   cmap=CMAP, norm=CNORM, s=22, edgecolors='none', zorder=2)
    best = best_allocations.loc[best_allocations.norm.eq(norm)].sort_values('K')
    ax.plot(best.K, best.mean_cooperation, color='#34383A', linewidth=1.15, zorder=3)
    ax.scatter(best.K, best.mean_cooperation, c=best.allocation_fraction, cmap=CMAP, norm=CNORM, s=38,
               edgecolors='#34383A', linewidths=.35, zorder=5)
    for row in best.itertuples():
        if row.K:
            color = tuple(np.asarray(CMAP(row.allocation_fraction)[:3]) * .72)
            ax.annotate(f'({row.k1},{row.k2})', (row.K, row.mean_cooperation), xytext=(0, 8), textcoords='offset points',
                        ha='center', va='bottom', fontsize=8, color=color)
    ax.set(xlim=(-.5, K_MAX + .5), ylim=(-.03, 1.04), xticks=np.arange(K_MAX + 1), yticks=[0, .5, 1],
           xlabel=r'Total depth $K=k_1+k_2$', ylabel=r'Mean cooperation $\bar\rho$', title=NORM_NAMES[norm])
    ax.spines[['top', 'right']].set_visible(False)
    if panel: ax.text(-.13, 1.06, panel, transform=ax.transAxes, fontweight='bold', fontsize=12)

def add_colorbar(fig, cax):
    bar = fig.colorbar(plt.cm.ScalarMappable(norm=CNORM, cmap=CMAP), cax=cax, ticks=[0, .5, 1])
    bar.ax.set_yticklabels(['0: strictness', '0.5: balanced', '1: tolerance'])
    bar.set_label(r'Allocation fraction $k_1/K$')

def save_figure(fig, name):
    for extension in ('pdf', 'svg'):
        fig.savefig(FIGURES / f'{name}.{extension}', bbox_inches='tight', pad_inches=.05)

In [5]:
for norm in NORM_ORDER:
    fig, ax = plt.subplots(figsize=(5.7, 3.7))
    fig.subplots_adjust(left=.13, right=.73, bottom=.24, top=.87)
    draw_panel(ax, norm)
    add_colorbar(fig, fig.add_axes([.77, .29, .025, .49]))
    fig.legend(handles=HANDLES, loc='lower center', bbox_to_anchor=(.46, .02), ncol=2, frameon=False, fontsize=8)
    save_figure(fig, f'panel_{norm}')
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 7.4), sharex=True, sharey=True)
fig.subplots_adjust(left=.08, right=.81, bottom=.16, top=.93, wspace=.20, hspace=.38)
for index, (ax, norm) in enumerate(zip(axes.flat, NORM_ORDER)):
    draw_panel(ax, norm, 'abcd'[index])
    if index % 2: ax.set_ylabel('')
    if index < 2: ax.set_xlabel('')
add_colorbar(fig, fig.add_axes([.85, .26, .016, .48]))
fig.legend(handles=HANDLES, loc='lower center', bbox_to_anchor=(.45, .035), ncol=2, frameon=False, fontsize=9)
save_figure(fig, 'Figure4_combined')
plt.show(); plt.close(fig)
assert len(list(FIGURES.glob('*.pdf'))) == len(list(FIGURES.glob('*.svg'))) == 5
print('Saved four panels and one combined figure in PDF and SVG.')

Saved four panels and one combined figure in PDF and SVG.
